<div style='background: #0f1f3d; padding: 40px 48px 36px; border-radius: 6px;'>
  <p style='color: #5a9bd5; margin: 0 0 16px 0; font-size: 0.72em; letter-spacing: 3px; text-transform: uppercase; font-weight: 500;'>FARMSA Research Group &mdash; Module [X]</p>
  <h1 style='color: #ffffff; font-size: 2em; font-weight: 600; margin: 0 0 6px 0; letter-spacing: -0.5px;'>[Estimator Name]</h1>
  <p style='color: #8fadc8; font-size: 0.95em; font-weight: 400; margin: 12px 0 0 0;'>Author: [Your Name]</p>
</div>

> **How to use this template:**  Copy this file, rename it `M1_ledoit_wolf.ipynb` (or whichever module you're working on), and fill in the sections below. The backtest code already works — you only need to write `estimate_covariance()`. Everything benchmarks against the sample covariance baseline from the preface.

---
## §1 &nbsp; Motivation &amp; Derivation

<!-- 
WHAT TO WRITE HERE:

1. Motivation (~150 words)
   - What's wrong with using the raw sample covariance?
   - What does your estimator fix?
   - When would you expect it to work well / poorly?

2. Derivation
   - Walk through the math from first principles.
   - Show the key equations step by step.
   - End with the closed-form formula or algorithm.
-->

$$
\hat{\Sigma} = \;\text{[your estimator here]}
$$

In [ ]:
# ── §2  Setup & Estimator ──────────────────────────────────
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from scipy.optimize import minimize
warnings.filterwarnings('ignore')

# ── Load the shared universe (same 50 stocks for every module) ──
prices  = pd.read_csv('data/prices.csv',  index_col=0, parse_dates=True)
returns = pd.read_csv('data/returns.csv', index_col=0, parse_dates=True)
N, T = returns.shape[1], returns.shape[0]
print(f"Loaded {N} assets × {T} days")


# ╔══════════════════════════════════════════════════════════╗
# ║  YOUR WORK GOES HERE                                    ║
# ║                                                         ║
# ║  Replace the body of this function with your estimator. ║
# ║  Input:  returns_df  (pd.DataFrame, T×N daily returns)  ║
# ║  Output: N×N numpy array (symmetric, positive semi-def) ║
# ╚══════════════════════════════════════════════════════════╝

def estimate_covariance(returns_df):
    """Return an N×N covariance matrix using [YOUR METHOD]."""
    # placeholder — just the sample covariance
    return returns_df.cov().values


# ── Quick sanity check (don't remove) ──
cov_test = estimate_covariance(returns.iloc[:252])
assert cov_test.shape == (N, N), f"Expected ({N},{N}), got {cov_test.shape}"
assert np.allclose(cov_test, cov_test.T), "Not symmetric"
assert np.all(np.linalg.eigvalsh(cov_test) > -1e-10), "Negative eigenvalues"
print("✓ Estimator passes sanity checks")

In [ ]:
# ── §3  Backtest ───────────────────────────────────────────
# This cell is ready to go — no changes needed.
# It runs a rolling monthly backtest comparing three strategies:
#   1. Equal Weight          (1/N — naive baseline)
#   2. Sample Cov MVO        (min-variance using raw sample covariance)
#   3. Your Estimator MVO    (min-variance using YOUR estimate_covariance)

def min_variance_portfolio(cov):
    """Minimum-variance portfolio (long-only, fully invested)."""
    n = cov.shape[0]
    w0 = np.ones(n) / n
    res = minimize(lambda w: w @ cov @ w, w0, method='SLSQP',
                   bounds=[(0, 1)] * n,
                   constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}],
                   options={'ftol': 1e-12, 'maxiter': 1000})
    return res.x if res.success else w0

LOOKBACK = 252   # 1 year estimation window
REBAL    = 21    # rebalance every ~1 month
MIN_HIST = 252   # need at least 1 year before first trade

dates = returns.index[MIN_HIST:]
n = returns.shape[1]

pv = {'Equal Weight': [1.0], 'Sample Cov MVO': [1.0], 'Your Estimator MVO': [1.0]}
cw = {k: np.ones(n) / n for k in pv}

for i, date in enumerate(dates):
    idx = returns.index.get_loc(date)
    if i % REBAL == 0:
        window = returns.iloc[idx - LOOKBACK : idx]
        cw['Equal Weight']       = np.ones(n) / n
        cw['Sample Cov MVO']     = min_variance_portfolio(window.cov().values)
        cw['Your Estimator MVO'] = min_variance_portfolio(estimate_covariance(window))

    day_ret = returns.iloc[idx].values
    for k in pv:
        pv[k].append(pv[k][-1] * (1 + cw[k] @ day_ret))

portfolio_df = pd.DataFrame(pv, index=[dates[0] - pd.Timedelta(days=1)] + list(dates))
print(f"✓ Backtest complete — {len(dates)} trading days")

In [ ]:
# ── §4  Results ────────────────────────────────────────────
C = {'Equal Weight':'#888','Sample Cov MVO':'#2c5282','Your Estimator MVO':'#c53030'}

fig, ax = plt.subplots(figsize=(11, 4.5))
for k in portfolio_df: ax.plot(portfolio_df.index, portfolio_df[k], label=k, color=C[k], lw=1.8)
ax.set_ylabel('Value ($1 start)'); ax.set_title('Cumulative Performance', fontweight='bold')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.2); plt.tight_layout(); plt.show()

# Metrics
def metrics(v):
    r = v.pct_change().dropna()
    ar, av = r.mean()*252, r.std()*np.sqrt(252)
    return {'Ret%': ar*100, 'Vol%': av*100, 'Sharpe': ar/av if av>0 else 0,
            'MaxDD%': ((v/v.cummax())-1).min()*100}
display(pd.DataFrame({k: metrics(portfolio_df[k]) for k in portfolio_df}).T.round(2))

---
## §5 &nbsp; Interpretation

<!-- ~200 words. Beat the baseline? When/why? Limitations? One-sentence takeaway. -->

---
<p style='text-align:center; color:#8fadc8; font-style:italic; font-size:0.85em;'>
Module [X] complete &mdash; see integration notebook for side-by-side comparison across all estimators.
</p>